# Risk Scoring
Menentukan **Risk Tier** vendor dari dua faktor:
- **Faktor operasional**: prediksi model dari notebook 03. Kurang baik jika `Risk_Score` ≥ `operational_threshold`.
- **Faktor teknis**: aturan per metrik (`On_Time_Delivery_Rate`, `Defect_Rate`, `Lead_Time_Days`, `Previous_Disruptions`). Kurang baik jika gagal minimal `min_tech_fails` kriteria.

| | Teknis baik | Teknis kurang baik |
|---|---|---|
| **Operasional baik** | Rendah | Sedang |
| **Operasional kurang baik** | Sedang | Tinggi |

Logika penentuan tier ada di `app/risk_scoring.py` agar sama persis dengan yang dipakai API.

# Import Library

In [1]:
import os
import sys
import json
import pandas as pd
import joblib

sys.path.append('..')
from app.risk_scoring import assign_risk_tier

# Data & Model Loading

In [2]:
pipeline = joblib.load('../models/v1/model_pipeline.pkl')

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv')['Risk_Level']
id_test = pd.read_csv('../data/processed/id_test.csv')['Supplier_ID']

# Konfigurasi Kriteria
Batas kriteria teknis diambil dari sekitar kuartil data train sebagai titik awal, lalu sesuaikan dengan standar bisnis. Median dihitung dari data train untuk mengisi nilai kosong pada faktor teknis.

In [3]:
TECH_FEATURES = ['On_Time_Delivery_Rate', 'Defect_Rate', 'Lead_Time_Days', 'Previous_Disruptions']

X_train[TECH_FEATURES].describe().round(2)

,On_Time_Delivery_Rate,Defect_Rate,Lead_Time_Days,Previous_Disruptions
count,19158.00,19099.00,19150.00,19156.00
mean,84.29,2.17,31.37,1.04
std,9.57,1.75,18.10,1.01
min,50.00,0.00,7.00,0.00
25%,77.73,0.64,16.00,0.00
50%,84.50,2.00,30.00,1.00
75%,91.45,3.38,44.00,2.00
max,100.00,10.00,120.00,5.00


In [4]:
config = {
    'operational_threshold': 0.5,
    'min_tech_fails': 2,
    'tech_rules': {
        'On_Time_Delivery_Rate': {'op': '<', 'value': 80},
        'Defect_Rate': {'op': '>', 'value': 3.5},
        'Lead_Time_Days': {'op': '>', 'value': 35},
        'Previous_Disruptions': {'op': '>=', 'value': 2},
    },
    'tech_medians': X_train[TECH_FEATURES].median().to_dict(),
}

# Penentuan Risk Tier pada Data Test

In [5]:
hasil = assign_risk_tier(pipeline, X_test, config)
hasil.insert(0, 'Supplier_ID', id_test)

hasil.head(10)

c:\Users\HP\anaconda3\envs\capstone_project\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\HP\anaconda3\envs\capstone_project\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\HP\anaconda3\envs\capstone_project\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\anaconda3\envs\capstone_project\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\HP\anaconda3\envs\capstone_pr

,Supplier_ID,Risk_Score,Operasional_Buruk,Teknis_Buruk,Kriteria_Teknis_Gagal,Risk_Tier
0,SUP_30926,0.890131,True,False,Previous_Disruptions,Sedang
1,SUP_19820,0.971281,True,True,"Defect_Rate, Lead_Time_Days",Tinggi
2,SUP_26561,0.093006,False,True,"On_Time_Delivery_Rate, Lead_Time_Days",Sedang
3,SUP_15014,0.027974,False,False,,Rendah
4,SUP_25378,0.954986,True,False,,Sedang
5,SUP_07838,0.678871,True,False,Lead_Time_Days,Sedang
6,SUP_07463,0.982991,True,False,Lead_Time_Days,Sedang
7,SUP_25546,0.618055,True,False,Lead_Time_Days,Sedang
8,SUP_15763,0.415961,False,True,"Lead_Time_Days, Previous_Disruptions",Sedang
9,SUP_25891,0.989775,True,False,,Sedang


In [6]:
hasil['Risk_Tier'].value_counts(normalize=True).sort_index().round(3)

Risk_Tier
Rendah    0.251
Sedang    0.509
Tinggi    0.240
Name: proportion, dtype: float64

## Proporsi Gagal per Kriteria Teknis

In [7]:
gagal = hasil['Kriteria_Teknis_Gagal'].str.split(', ').explode()
(gagal[gagal != ''].value_counts() / len(hasil)).round(3)

Kriteria_Teknis_Gagal
Lead_Time_Days           0.384
On_Time_Delivery_Rate    0.319
Previous_Disruptions     0.276
Defect_Rate              0.221
Name: count, dtype: float64

## Validasi terhadap Label Asli
Semakin tinggi tier, seharusnya semakin banyak vendor yang sebenarnya berlabel `Risk_Level = 1` (berisiko).

In [8]:
pd.crosstab(hasil['Risk_Tier'], y_test, normalize='index').round(3)

Risk_Level,0,1
Risk_Tier,,
Rendah,0.715,0.285
Sedang,0.210,0.790
Tinggi,0.057,0.943


## Contoh Hasil per Tier

In [9]:
hasil.groupby('Risk_Tier', observed=True).head(2).sort_values('Risk_Tier')

,Supplier_ID,Risk_Score,Operasional_Buruk,Teknis_Buruk,Kriteria_Teknis_Gagal,Risk_Tier
3,SUP_15014,0.027974,False,False,,Rendah
11,SUP_07817,0.381086,False,False,,Rendah
0,SUP_30926,0.890131,True,False,Previous_Disruptions,Sedang
2,SUP_26561,0.093006,False,True,"On_Time_Delivery_Rate, Lead_Time_Days",Sedang
1,SUP_19820,0.971281,True,True,"Defect_Rate, Lead_Time_Days",Tinggi
12,SUP_33191,0.999478,True,True,"On_Time_Delivery_Rate, Previous_Disruptions",Tinggi


# Simpan Konfigurasi

In [10]:
os.makedirs('../models', exist_ok=True)

with open('../models/v1/risk_tier_config.json', 'w') as f:
    json.dump(config, f, indent=2)